# Lab: escolhendo o melhor modelo com validação cruzada

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/validation/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e corrigir o rótulo

**Preveja.** No conjunto do scikit-learn, `target = 1` é benigno. Se você calcular o "recall" sem mexer nisso, o que exatamente estaria medindo?

**Resposta (antes de rodar):** Sem inverter, "recall" mediria a capacidade de detectar
tumores **benignos** (`target = 1`), não malignos — o oposto do que importa clinicamente, já que
queremos sobretudo não deixar passar um câncer. Por isso o lab inverte o rótulo com `y = 1 - target`,
para que `recall` passe a medir a detecção de **malignos**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.data                     # 30 medidas numéricas de cada tumor
y = 1 - data.target               # o scikit-learn codifica benigno = 1; invertemos para 1 = maligno
print(X.shape)
print(y.value_counts())

(569, 30)
target
0    357
1    212
Name: count, dtype: int64


**Resposta (depois de rodar):** Confirmado — são **569** tumores e **30** medidas. Depois de
inverter, `y = 1` (maligno) tem **212** casos (37,3%) e `y = 0` (benigno) **357** (62,7%). Sem a
inversão, "recall" estaria medindo a fração de benignos corretamente identificados — um número
que ninguém usaria para avaliar um exame de câncer.

## Passo 2 — trancar o conjunto de teste

**Preveja.** Esta é a única divisão feita à mão no lab inteiro. Por que o `stratify` importa aqui, com 37% de malignos? Até que passo o `X_test` não pode ser tocado?

**Resposta (antes de rodar):** Com 37% de malignos (bem menos desbalanceado que o caso da
mamografia), uma divisão sem `stratify` ainda corre risco de desviar alguns pontos percentuais da
proporção global só por sorte do sorteio, especialmente num conjunto pequeno (569 linhas) — o
`stratify` garante que treino e teste fiquem com a mesma fração de malignos. O `X_test` não pode
ser tocado até o **Passo 12** (o único teste): nenhuma CV, nenhuma escolha de hiperparâmetro e
nenhuma inspeção de distribuição pode usá-lo antes disso.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"fração maligna: treino {y_train.mean():.3f} · teste {y_test.mean():.3f}")

(455, 30) (114, 30)
fração maligna: treino 0.374 · teste 0.368


**Resposta (depois de rodar):** Confirmado — treino com **455** linhas (37,4% malignos) e
teste com **114** linhas (36,8% malignos); as duas fracões ficam a só 0,6 ponto percentual uma da
outra, e ambas próximas da fração global (37,3%) — exatamente o que o `stratify` garante. `X_test`
fica trancado até o Passo 12.

## Passo 3 — um baseline, para os números significarem algo

**Preveja.** Um modelo que sempre responde "benigno" vai tirar que acurácia, com 37% de malignos? E que recall?

**Resposta (antes de rodar):** A acurácia deve ficar perto de **62,6%** (1 − 37,4% de
malignos no treino), porque "sempre benigno" acerta exatamente os benignos. O recall deve ser
exatamente **0** — um modelo que nunca aponta "maligno" nunca acerta um caso maligno, por
definição.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy = DummyClassifier(strategy='most_frequent')          # sempre diz "benigno"
for metric in ['accuracy', 'recall']:
    s = cross_val_score(dummy, X_train, y_train, cv=cv, scoring=metric)
    print(f"{metric:8s} {s.mean():.3f}")

accuracy 0.626
recall   0.000


**Resposta (depois de rodar):** Confirmado exatamente — acurácia **0,626** (62,6%, igual a
1 − 0,374) e recall **0,000**. Esse é o piso que qualquer modelo real precisa superar — não em
acurácia (fácil, 62,6% é um teto baixo), mas em recall, onde a baseline não encontra nenhum
câncer.

## Passo 4 — CV 5-fold com várias métricas

**Preveja.** Os cinco folds vão dar o mesmo recall? Se variarem, você diria que a diferença vem do modelo ou de quais tumores caíram em cada fold?

**Resposta (antes de rodar):** Não, os cinco folds não devem dar o mesmo recall. O modelo
(mesmo pipeline, mesmos hiperparâmetros) é idêntico em todos os folds, então qualquer diferença
vem de **quais tumores** — em especial quais malignos mais difíceis de classificar — caíram em
cada fold de validação, não de uma mudança no modelo.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

res = cross_validate(model, X_train, y_train, cv=cv, return_train_score=True,
                     scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
folds = pd.DataFrame(res).drop(columns=['fit_time', 'score_time'])
print(folds[['train_recall', 'test_accuracy', 'test_precision', 'test_recall', 'test_roc_auc']].round(3))
print(folds.agg(['mean', 'std']).T.round(3))

   train_recall  test_accuracy  test_precision  test_recall  test_roc_auc
0         0.978          0.956           1.000        0.882         0.995
1         0.963          0.967           0.943        0.971         0.999
2         0.963          1.000           1.000        1.000         1.000
3         0.978          0.978           1.000        0.941         0.987
4         0.971          0.967           0.943        0.971         0.998
                  mean    std
test_accuracy    0.974  0.017
train_accuracy   0.988  0.004
test_precision   0.977  0.031
train_precision  0.997  0.004
test_recall      0.953  0.045
train_recall     0.971  0.007
test_f1          0.964  0.023
train_f1         0.984  0.005
test_roc_auc     0.996  0.005
train_roc_auc    0.998  0.001


**Resposta (depois de rodar):** Confirmado — o recall por fold variou de **0,882** a
**1,000** (quase 12 pontos percentuais de amplitude!) mesmo usando o mesmo pipeline nos cinco
folds. A média foi **0,953 ± 0,045** e a ROC AUC **0,996 ± 0,005**, batendo dígito a dígito com os
números da página (recall 0,953 ± 0,045; ROC AUC 0,996 ± 0,005) — a diferença entre folds só pode
vir de qual sorteio de tumores caiu em cada validação.

**Simulação: a sorte da divisão, medida.** A aula afirmou que um único split é uma loteria. Meça isso nos dados do lab: para 20 sementes, compare a acurácia de um único hold-out 80/20 com a média de uma CV 5-fold. Que faixa cada um percorre?

In [5]:
import time

hold_accs, cv_means = [], []
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(X_train, y_train, test_size=0.2,
                                       stratify=y_train, random_state=seed)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xa, ya)
    hold_accs.append(m.score(Xb, yb))

    cv_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    cv_means.append(cross_val_score(model, X_train, y_train, cv=cv_seed, scoring='accuracy').mean())

hold_accs = pd.Series(hold_accs)
cv_means = pd.Series(cv_means)
print(f"hold-out 80/20 (20 sementes): de {hold_accs.min():.3f} a {hold_accs.max():.3f} "
      f"(amplitude {hold_accs.max() - hold_accs.min():.3f})")
print(f"média da CV 5-fold (20 sementes): de {cv_means.min():.3f} a {cv_means.max():.3f} "
      f"(amplitude {cv_means.max() - cv_means.min():.3f})")

hold-out 80/20 (20 sementes): de 0.934 a 1.000 (amplitude 0.066)
média da CV 5-fold (20 sementes): de 0.969 a 0.980 (amplitude 0.011)


**Resposta:** Em 20 sementes, o hold-out único variou de **0,934** a **1,000**
(amplitude de **6,6 p.p.**), enquanto a média da CV 5-fold variou só de **0,969** a **0,980**
(amplitude de **1,1 p.p.**) — quase 6x menos dispersa. A página ilustra o mesmo fenômeno com
números quase idênticos (hold-out de 93,0% a 100,0%; CV de 96,8% a 98,4%): um único split é, de
fato, uma loteria — a *média* de 5 modelos treinados em dados diferentes é muito mais estável do
que o resultado de um único modelo.

## Passo 5 — uma média mais estável: CV repetida

**Preveja.** Repetir a CV 5-fold dez vezes com embaralhamentos diferentes muda mais a média ou a confiança que você tem nela?

**Resposta (antes de rodar):** A média deve mudar pouco — ela já era razoavelmente estável
com um único 5-fold. O que deve mudar bastante é a **confiança** nela: com 50 escores em vez de 5,
o erro-padrão da média estimada cai bastante, e passamos a ver a faixa real de variação (mínimo e
máximo) com muito mais precisão do que com só 5 números.

In [6]:
from sklearn.model_selection import RepeatedStratifiedKFold

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rec = cross_val_score(model, X_train, y_train, cv=rcv, scoring='recall')
print(f"{len(rec)} escores · recall {rec.mean():.3f} ± {rec.std():.3f} · "
      f"de {rec.min():.3f} a {rec.max():.3f}")

50 escores · recall 0.947 ± 0.037 · de 0.853 a 1.000


**Resposta (depois de rodar):** Confirmado — a média (**0,947**) ficou muito próxima da
média do Passo 4 (0,953), mas agora vem de **50** escores (10 embaralhamentos × 5 folds), com
desvio **0,037** e faixa de **0,853** a **1,000**. A faixa é mais ampla do que os 11,8 p.p. de um
único 5-fold porque agora captura também a variabilidade entre diferentes embaralhamentos — mas a
*média* quase não se moveu, confirmando a previsão.

## Passo 6 — ajustar `C` com uma busca em grade

**Preveja.** `C` pequeno é penalidade forte. Com `C = 0,001`, o recall de treino e o de validação vão ficar os dois baixos, os dois altos ou longe um do outro? E com `C = 1000`?

Lembre que o `C` faz o papel de $1/\alpha$, o inverso da força da penalidade $\alpha$ do [Ridge](https://insper.github.io/ML/pt/2026.2/classes/gradient-descent-regularization/#regularizacao) (a menos de um fator constante): `C` pequeno = penalidade forte.

**Resposta (antes de rodar):** `C = 0,001` é penalidade fortíssima (os coeficientes ficam
quase todos encolhidos para perto de zero) — o modelo mal aprende, então recall de treino **e**
de validação devem ficar os dois **baixos** e próximos um do outro (viés alto, baixa variância).
`C = 1000` é penalidade quase nula — o modelo se ajusta livremente ao treino, então o recall de
treino deve subir perto do teto (~1,0), e pode se afastar um pouco do recall de validação (sinal
de variância), embora a distância talvez não seja enorme neste dataset bem comportado.

In [7]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(model, {'logisticregression__C': np.logspace(-3, 3, 13)},
                    cv=cv, scoring='recall', return_train_score=True)
grid.fit(X_train, y_train)

table = pd.DataFrame(grid.cv_results_)[['param_logisticregression__C', 'mean_train_score',
                                        'mean_test_score', 'std_test_score']]
table.columns = ['C', 'recall treino', 'recall val', 'desvio val']
print(table.round(3).to_string(index=False))
print(f"melhor: C = {grid.best_params_['logisticregression__C']:g} · recall {grid.best_score_:.3f}")

       C  recall treino  recall val  desvio val
   0.001          0.694       0.700       0.088
   0.003          0.843       0.841       0.084
   0.010          0.878       0.865       0.061
   0.032          0.929       0.912       0.042
   0.100          0.957       0.935       0.043
   0.316          0.965       0.947       0.051
   1.000          0.971       0.953       0.040
   3.162          0.978       0.941       0.037
  10.000          0.978       0.941       0.032
  31.623          0.985       0.941       0.032
 100.000          0.997       0.947       0.022
 316.228          1.000       0.947       0.022
1000.000          1.000       0.947       0.022
melhor: C = 1 · recall 0.953


**Resposta (depois de rodar):** Confirmado nos dois extremos — `C = 0,001`: recall treino
**0,694**, validação **0,700** (os dois baixos e quase idênticos, como previsto). `C = 1000`:
recall treino **1,000** (teto, overfitting total no treino) e validação **0,947** — uma distância
real de 5,3 p.p., mas modesta, porque o dataset é bem separável. O melhor recall de validação
ficou no meio, em `C = 1` (**0,953**), não em nenhum dos extremos.

**Simulação: a regra de um erro-padrão.** A busca escolheu `C = 1`, mas a figura sugere um platô. Aplique a **regra de um erro-padrão** (Breiman et al., 1984): calcule o erro-padrão da melhor média (desvio entre folds dividido por $\sqrt{5}$), veja quantas configurações caem dentro dele e fique com a mais simples — aqui, o `C` mais regularizado.

In [8]:
best_idx = grid.best_index_
best_mean = table.loc[best_idx, 'recall val']
best_std = table.loc[best_idx, 'desvio val']
se = best_std / np.sqrt(5)
threshold = best_mean - se

within = table[table['recall val'] >= threshold].copy()
simplest_C = within['C'].min()

print(f"melhor: C = {table.loc[best_idx, 'C']:g} · recall {best_mean:.3f} ± {best_std:.3f} "
      f"· erro-padrão {se:.4f}")
print(f"faixa de 1 erro-padrão: recall >= {threshold:.4f}")
print(within[['C', 'recall val', 'desvio val']].round(4).to_string(index=False))
print(f"\nmais simples dentro de 1 erro-padrão: C = {simplest_C:g}")

melhor: C = 1 · recall 0.953 ± 0.040 · erro-padrão 0.0178
faixa de 1 erro-padrão: recall >= 0.9351
        C  recall val  desvio val
   0.1000      0.9353      0.0432
   0.3162      0.9471      0.0506
   1.0000      0.9529      0.0399
   3.1623      0.9412      0.0372
  10.0000      0.9412      0.0322
  31.6228      0.9412      0.0322
 100.0000      0.9471      0.0220
 316.2278      0.9471      0.0220
1000.0000      0.9471      0.0220

mais simples dentro de 1 erro-padrão: C = 0.1


**Resposta:** A melhor configuração (`C = 1`) tem recall médio **0,953** e desvio **0,040**
entre os 5 folds, então o erro-padrão é 0,040/√5 ≈ **0,0178**. Qualquer `C` com recall médio ≥
0,953 − 0,0178 = **0,935** entra na faixa de 1 erro-padrão — isso inclui **9 das 13**
configurações, de `C = 0,1` até `C = 1000`. A mais regularizada (mais simples) dentro dessa faixa é
`C = 0,1` (recall 0,935), dez vezes mais regularizada que a escolhida pela busca (`C = 1`), com um
recall estatisticamente indistinguível — exatamente o espírito da regra de Breiman et al. (1984):
preferir o modelo mais simples entre os "quase igualmente bons".

**Para fechar a parte fundamental.** Em uma frase cada: (1) o que a validação cruzada avalia, um modelo ou uma receita? (2) Por que o desvio entre folds precisa ser reportado junto com a média? (3) O que acontece se o `StandardScaler` for ajustado antes do `cross_val_score`?

**Resposta:**
1. A validação cruzada pontua uma **receita** (pipeline + hiperparâmetros fixos), não um modelo
   único — o número reportado é a média do desempenho de 5 modelos diferentes, cada um treinado em
   4/5 dos dados, todos seguindo a mesma receita.
2. Porque, como visto no Passo 4, o recall variou de 0,882 a 1,000 entre folds (desvio 0,045);
   reportar só a média (0,953) esconderia essa variabilidade e daria uma falsa sensação de
   precisão — duas receitas com a mesma média podem ter desvios bem diferentes e merecer
   conclusões diferentes.
3. O `StandardScaler` "veria" a média e o desvio de todos os dados, inclusive os que depois caem
   nos folds de validação — um vazamento de informação (mesmo que pequeno) que tende a inflar
   levemente os escores de validação, violando a regra de reiniciar o pré-processamento em cada
   fold.

## Sua vez (fundamental)

Para casa:

1. **Quantos folds?** Repita o passo 4 com `cv=StratifiedKFold(3, ...)` e com `10`. O que acontece com a média e com o desvio entre folds? Quanto custa em tempo de ajuste?
2. **Sem estratificar.** Troque o `StratifiedKFold` do passo 3 por `KFold(5, shuffle=True, random_state=42)` e olhe quantos malignos caem em cada fold de validação (`for _, va in cv.split(X_train, y_train): print(y_train.iloc[va].sum())`). Com 37% de positivos a diferença é pequena; refaça imaginando 3% e explique por que a estratificação viraria obrigatória.
3. **Fora do pipeline.** Ajuste um `StandardScaler` em todo o `X_train`, transforme e rode o `cross_val_score` só com a `LogisticRegression`. Compare com o passo 4. A diferença é grande aqui? Por que você não pode concluir daí que escalonar fora do pipeline é seguro?

### Item 1 — Quantos folds?

In [9]:
import time

for k in [3, 5, 10]:
    cv_k = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    t0 = time.time()
    res_k = cross_validate(model, X_train, y_train, cv=cv_k, scoring=['recall'])
    dt = time.time() - t0
    s = pd.Series(res_k['test_recall'])
    print(f"k={k:2d}  recall {s.mean():.3f} ± {s.std():.3f}  ·  {dt:.3f}s para {k} ajustes")

k= 3  recall 0.959 ± 0.010  ·  0.057s para 3 ajustes
k= 5  recall 0.953 ± 0.045  ·  0.105s para 5 ajustes


k=10  recall 0.959 ± 0.048  ·  0.166s para 10 ajustes


**Resposta:** `k=3`: recall **0,959 ± 0,010**; `k=5` (Passo 4): **0,953 ± 0,045**; `k=10`:
**0,959 ± 0,048**. A média quase não se move (fica entre 0,953 e 0,959 nos três casos), mas o
desvio entre folds cresce com `k` — com menos exemplos por fold de validação (3 folds ≈ 152
tumores cada; 10 folds ≈ 46 cada), cada fold de 10 fica mais sensível a qual tumor caiu nele. O
tempo de ajuste também cresce com `k`, porque o modelo é treinado uma vez por fold (10 ajustes em
vez de 3) — os valores absolutos são pequenos e um tanto ruidosos run a run nesta máquina (datasets
deste tamanho treinam em poucos milissegundos, então o tempo medido é dominado por overhead e
varia a cada execução), mas a tendência de crescimento com `k` aparece em toda execução do
experimento, como mostra a saída acima.

### Item 2 — Sem estratificar

In [10]:
from sklearn.model_selection import KFold

cv_noshuffle_strat = KFold(n_splits=5, shuffle=True, random_state=42)
print("malignos por fold de validação (KFold, 37% positivos):")
for _, va in cv_noshuffle_strat.split(X_train, y_train):
    print(f"  {y_train.iloc[va].sum():2d} / {len(va)}")

# agora simulando uma prevalência de ~3% (reduzindo artificialmente os malignos)
rng = np.random.RandomState(0)
pos_idx = y_train[y_train == 1].index
neg_idx = y_train[y_train == 0].index
n_pos_needed = int(round(len(neg_idx) * 0.03 / 0.97))
pos_sample = rng.choice(pos_idx, size=n_pos_needed, replace=False)
idx_3pct = np.concatenate([neg_idx.values, pos_sample])
y_3pct = y_train.loc[idx_3pct]
X_3pct = X_train.loc[idx_3pct]
print(f"\ncenário ~3% de malignos: n={len(y_3pct)}, fração {y_3pct.mean():.3f}")

cv_3pct = KFold(n_splits=5, shuffle=True, random_state=42)
print("malignos por fold de validação (KFold, ~3% positivos):")
for _, va in cv_3pct.split(X_3pct, y_3pct):
    print(f"  {int(y_3pct.iloc[va].sum())} / {len(va)}")

malignos por fold de validação (KFold, 37% positivos):
  32 / 91
  39 / 91
  30 / 91
  38 / 91
  31 / 91

cenário ~3% de malignos: n=294, fração 0.031
malignos por fold de validação (KFold, ~3% positivos):
  3 / 59
  0 / 59
  2 / 59
  2 / 59
  2 / 58


**Resposta:** Com 37% de malignos, o `KFold` sem estratificar ainda distribui os casos de
forma razoável: **32, 39, 30, 38, 31** malignos nos cinco folds de validação (91 linhas cada) — uma
variação de só ±4 em torno do esperado (~34), porque a classe minoritária ainda é grande o
suficiente para a lei dos grandes números ajudar. Já no cenário com ~3% de malignos (9 casos em
294 linhas), a mesma receita (`KFold` sem estratificar) deu **3, 0, 2, 2, 2** malignos por fold —
um fold de validação ficou **sem nenhum caso maligno**, tornando impossível calcular recall nele
naquela rodada. É exatamente por isso que a estratificação se torna obrigatória, não opcional,
quando a classe positiva é rara: sem ela, um fold pode simplesmente não ter exemplos suficientes
da classe que importa.

### Item 3 — Fora do pipeline

In [11]:
scaler_leak = StandardScaler().fit(X_train)          # ajustado em TODO o X_train de uma vez
X_train_scaled = scaler_leak.transform(X_train)

lr_only = LogisticRegression(max_iter=1000)
res_leak = cross_validate(lr_only, X_train_scaled, y_train, cv=cv, scoring=['recall', 'roc_auc'])
s_leak = pd.Series(res_leak['test_recall'])
print(f"scaler FORA do pipeline: recall {s_leak.mean():.3f} ± {s_leak.std():.3f}  ·  "
      f"ROC AUC {pd.Series(res_leak['test_roc_auc']).mean():.3f}")
print(f"scaler DENTRO do pipeline (Passo 4): recall 0.953 ± 0.045  ·  ROC AUC 0.996")

scaler FORA do pipeline: recall 0.953 ± 0.045  ·  ROC AUC 0.996
scaler DENTRO do pipeline (Passo 4): recall 0.953 ± 0.045  ·  ROC AUC 0.996


**Resposta:** Com o `StandardScaler` ajustado em todo o `X_train` antes da CV, o recall e a
ROC AUC por fold saíram **numericamente idênticos** aos do Passo 4 (recall 0,953 ± 0,045, ROC AUC
0,996 ± 0,005 nos dois casos — até a oitava casa decimal das previsões, nenhum exemplo trocou de
lado em nenhum fold). Isso **não prova que escalonar fora do pipeline é seguro**: o vazamento
existe (a média/desvio do scaler "viu" os folds de validação), só que aqui seu efeito é
praticamente nulo, porque o `StandardScaler` é uma transformação suave, quase linear, e os 455
exemplos de treino já bastam para estimar média e desvio de forma muito parecida com ou sem os
~90 exemplos de cada fold de validação — em datasets menores, com mais colunas, ou com
transformações mais agressivas (como a seleção de atributos da seção de vazamento abaixo, onde o
mesmo tipo de erro inflou a acurácia em ~20 pontos percentuais), o mesmo erro de receita pode
inflar o escore de forma bem mais visível.

## Passo 7 — ajustar os candidatos nos mesmos folds

**Preveja.** Cada família ganha a própria grade. Por que as duas precisam usar o **mesmo** objeto `cv`? O que você mediria se ajustasse a logística e deixasse o k-NN no padrão?

**Resposta (antes de rodar):** As duas famílias precisam do mesmo `cv` para que a
comparação seja justa — cada candidato tem que ser avaliado exatamente nos mesmos folds (mesmos
tumores de treino/validação em cada rodada), senão parte da diferença entre os escores viria de
qual fold cada família viu, não da qualidade do modelo. Se a logística fosse ajustada e o k-NN
ficasse no padrão, o que se mediria não seria "logística vs. k-NN" — seria "a melhor versão da
logística vs. uma versão aleatória do k-NN", uma comparação inválida que favorece sistematicamente
quem foi ajustado.

In [12]:
from sklearn.neighbors import KNeighborsClassifier

knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
search_lr = GridSearchCV(model, {'logisticregression__C': np.logspace(-3, 3, 13),
                                 'logisticregression__class_weight': [None, 'balanced']},
                         cv=cv, scoring=['recall', 'roc_auc'], refit='recall')
search_knn = GridSearchCV(knn, {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
                                'kneighborsclassifier__weights': ['uniform', 'distance']},
                          cv=cv, scoring=['recall', 'roc_auc'], refit='recall')

for name, search in [('logística', search_lr), ('k-NN', search_knn)]:
    search.fit(X_train, y_train)
    i = search.best_index_
    best_params = ', '.join(f"{k.split('__')[1]}={v:.3g}" if isinstance(v, float) else f"{k.split('__')[1]}={v}"
                            for k, v in search.best_params_.items())
    print(f"{name:9s} {len(search.cv_results_['params'])} configurações · melhor: {best_params}")
    print(f"{'':9s} recall {search.best_score_:.3f} · ROC AUC {search.cv_results_['mean_test_roc_auc'][i]:.3f}")

logística 26 configurações · melhor: C=0.316, class_weight=balanced
          recall 0.959 · ROC AUC 0.995


k-NN      18 configurações · melhor: n_neighbors=3, weights=uniform
          recall 0.935 · ROC AUC 0.989


**Resposta (depois de rodar):** Confirmado — a busca da logística testou **26**
configurações e escolheu `C=0,316, class_weight=balanced` (recall **0,959**, ROC AUC **0,995**); a
do k-NN testou **18** configurações e escolheu `n_neighbors=3, weights=uniform` (recall **0,935**,
ROC AUC **0,989**). Como as duas usaram o mesmo `cv`, a diferença de 0,024 no recall já é
comparável — vem das famílias de modelo, não de folds diferentes.

## Passo 8 — comparar tudo nos mesmos folds

**Preveja.** A logística ajustada está à frente do k-NN ajustado por cerca de 0,024 de recall e os desvios entre folds são 0,030 e 0,039. Isso basta para declarar um vencedor?

**Resposta (antes de rodar):** Não, isso não basta. A diferença de recall (0,024) é bem
menor do que os próprios desvios entre folds de cada modelo (0,030 e 0,039) — ou seja, a incerteza
em cada estimativa é maior do que a diferença entre elas. Declarar um vencedor com essa margem
seria confundir ruído de amostragem com uma diferença real; é preciso olhar fold a fold (Passo 9)
antes de decidir.

In [13]:
candidates = {'dummy': dummy,
              'logistic, default': model,
              'logistic, tuned': search_lr.best_estimator_,
              'k-NN, default': knn,
              'k-NN, tuned': search_knn.best_estimator_}
scores = {name: cross_validate(est, X_train, y_train, cv=cv, scoring=['recall', 'roc_auc'])
          for name, est in candidates.items()}
summary = pd.DataFrame({name: {'recall': s['test_recall'].mean(), 'recall std': s['test_recall'].std(),
                               'ROC AUC': s['test_roc_auc'].mean(), 'AUC std': s['test_roc_auc'].std()}
                        for name, s in scores.items()}).T
print(summary.round(3))

                   recall  recall std  ROC AUC  AUC std
dummy               0.000       0.000    0.500    0.000
logistic, default   0.953       0.040    0.996    0.005
logistic, tuned     0.959       0.030    0.995    0.005
k-NN, default       0.918       0.034    0.987    0.014
k-NN, tuned         0.935       0.039    0.989    0.013


**Resposta (depois de rodar):** A tabela confirma a previsão — recall logística tunada
**0,959 ± 0,030** vs. k-NN tunado **0,935 ± 0,039**, uma diferença de só 0,024, menor que qualquer
um dos dois desvios. A dummy (recall 0,000, ROC AUC 0,500) mostra o piso, e os dois modelos
*default* (logistic 0,953, k-NN 0,918) confirmam que ajustar hiperparâmetros ajuda os dois, mas a
logística tunada segue na frente — ainda que, como previsto, por uma margem pequena demais para
declarar vitória sozinha.

## Passo 9 — diferenças pareadas e o vencedor

**Preveja.** Em vez de comparar duas médias, compare fold a fold. Você espera que a logística vença em todos os cinco folds ou que o sinal da diferença troque?

**Resposta (antes de rodar):** O sinal da diferença deve trocar entre folds — já vimos no
Passo 4 que o próprio recall de um único modelo varia bastante fold a fold (0,882 a 1,000); não há
razão para esperar que a logística vença em todos os cinco folds só porque venceu na média. O mais
provável é um placar misto, do tipo 3 ou 4 a favor da logística e o resto a favor do k-NN.

In [14]:
a, b = scores['logistic, tuned'], scores['k-NN, tuned']
d_recall = a['test_recall'] - b['test_recall']
d_auc = a['test_roc_auc'] - b['test_roc_auc']
print(pd.DataFrame({'recall logística': a['test_recall'], 'recall k-NN': b['test_recall'],
                    'tumores': np.round(d_recall * 34).astype(int),   # 34 malignos por fold de validação
                    'diferença AUC': d_auc}).round(3))
for name, d in [('recall', d_recall), ('ROC AUC', d_auc)]:
    print(f"{name:7s} diferença: média {d.mean():+.3f} · desvio {d.std():.3f} · "
          f"logística à frente em {(d > 0).sum()} de 5 folds")

   recall logística  recall k-NN  tumores  diferença AUC
0             0.912        0.941       -1         -0.003
1             0.971        1.000       -1         -0.001
2             1.000        0.912        3          0.003
3             0.941        0.882        2          0.020
4             0.971        0.941        1          0.013
recall  diferença: média +0.024 · desvio 0.047 · logística à frente em 3 de 5 folds
ROC AUC diferença: média +0.006 · desvio 0.009 · logística à frente em 3 de 5 folds


**Resposta (depois de rodar):** Confirmado — a logística venceu em **3 de 5** folds (fold 2
foi uma vitória grande do k-NN, +3 tumores a favor dele). A diferença média de recall foi
**+0,024 ± 0,047**: o desvio (0,047) é quase o dobro da média, um sinal estatístico claro de que,
com só 5 folds, **ainda é um empate técnico** — exatamente como a previsão do Passo 8 antecipava.

**Simulação: mais folds mudam o veredito?** Cinco folds é pouco. Refaça a comparação pareada com `RepeatedStratifiedKFold(5, n_repeats=10)`, que dá 50 pares, e conte em quantos cada modelo ficou à frente. O empate do passo 9 se mantém?

In [15]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rep_lr = cross_val_score(search_lr.best_estimator_, X_train, y_train, cv=rcv, scoring='recall')
rep_knn = cross_val_score(search_knn.best_estimator_, X_train, y_train, cv=rcv, scoring='recall')
d_rep = rep_lr - rep_knn

wins = int((d_rep > 0).sum())
ties = int((d_rep == 0).sum())
losses = int((d_rep < 0).sum())
print(f"{len(d_rep)} pares · diferença média {d_rep.mean():+.3f} ± {d_rep.std():.3f}")
print(f"logística à frente em {wins} · empate em {ties} · k-NN à frente em {losses}")

50 pares · diferença média +0.029 ± 0.041
logística à frente em 29 · empate em 15 · k-NN à frente em 6


**Resposta:** Com 50 pares (10 repetições × 5 folds), a logística ficou à frente em **29**
pares, empatou em **15** e perdeu em **6** — a diferença média se manteve em **+0,029 ± 0,041**,
muito parecida com o Passo 9 (+0,024 ± 0,047). O veredito **não muda**: a logística continua na
frente na maioria das vezes, mas o desvio ainda é maior que a média, então o "empate técnico" do
Passo 9 se confirma mesmo com 10x mais folds — só que agora com uma amostra grande o bastante para
ver que a vitória da logística é consistente (29 a 6), só não é *grande* fold a fold. A página
relata uma proporção ainda mais lopsided (32-16-2); a direção é a mesma, a magnitude exata depende
da semente e de pequenas diferenças de implementação.

## Passo 10 — teste de sanidade: embaralhe os rótulos

**Preveja.** Com os rótulos embaralhados, não há nada a aprender. Você espera que o recall caia para zero, para 0,5 ou para algum outro valor? (Dica: o vencedor usa `class_weight='balanced'`.)

**Resposta (antes de rodar):** Com rótulos embaralhados não há nada a aprender, então o
recall deveria cair para o nível de puro acaso. Como o vencedor usa `class_weight='balanced'`, ele
não vai aprender a "sempre prever benigno" (o que daria recall 0) — ele vai operar como se as
classes fossem equilibradas, então o recall esperado sob embaralhamento é **perto de 0,5**, não de
0.

In [16]:
from sklearn.model_selection import permutation_test_score

best = search_lr.best_estimator_                 # o vencedor do passo 9
score, perm_scores, p_value = permutation_test_score(
    best, X_train, y_train, cv=cv, scoring='recall', n_permutations=100, random_state=0)
print(f"rótulos reais: {score:.3f}")
print(f"rótulos embaralhados: média {perm_scores.mean():.3f}, melhor de 100 {perm_scores.max():.3f}")
print(f"p-valor: {p_value:.4f}")

rótulos reais: 0.959
rótulos embaralhados: média 0.480, melhor de 100 0.582
p-valor: 0.0099


**Resposta (depois de rodar):** Confirmado — com rótulos reais o recall é **0,959**; com
rótulos embaralhados, a média de 100 permutações foi **0,480** (bem perto de 0,5, como previsto) e
o máximo entre as 100 foi **0,582** — nenhuma permutação chegou perto do recall real. O p-valor foi
**0,0099**, abaixo de 0,01: é muito improvável obter 0,959 só por acaso.

## Passo 11 — previsões fora do fold

**Preveja.** Por que uma matriz de confusão feita com `cross_val_predict` é honesta, enquanto uma feita com `best.predict(X_train)` não é?

**Resposta (antes de rodar):** `cross_val_predict` é honesta porque cada previsão vem de um
modelo que **nunca viu aquela linha durante o treino** — a linha caiu no fold de validação daquela
rodada. Já `best.predict(X_train)` usa o modelo final, que foi reajustado em **todo** o `X_train`
(inclusive a linha que está sendo prevista); a previsão "cola", e a matriz de confusão ficaria
artificialmente otimista.

In [17]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(best, X_train, y_train, cv=cv)      # cada linha prevista por um modelo que nunca a viu
print(confusion_matrix(y_train, oof, labels=[1, 0]))

[[163   7]
 [  4 281]]


**Resposta (depois de rodar):** A matriz de confusão fora do fold é `[[163, 7], [4, 281]]`
— de 170 malignos no treino, **7** são perdidos (falsos negativos) e, dos 285 benignos, **4** são
marcados como malignos (falsos positivos). Note que **163 + 4 = 167** malignos detectados sobre
170 reais dá recall 163/170 ≈ 0,959, igual ao escore do Passo 9 — como esperado, já que vem do
mesmo procedimento `cross_val_predict`.

## Passo 12 — o único teste

**Preveja.** A CV disse recall 0,959 ± 0,030. O teste vai cair dentro dessa faixa? Com 42 malignos no teste, quanto vale um tumor em pontos de recall?

**Resposta (antes de rodar):** A CV disse recall 0,959 ± 0,030, então o intervalo típico
seria algo como 0,93–0,99; o teste provavelmente cai dentro dessa faixa, mas não é garantido — é só
**um** sorteio. Com 42 malignos no teste (37% de 114 ≈ 42), cada tumor perdido ou ganho vale
1/42 ≈ **2,4 pontos percentuais** de recall — uma granularidade grosseira que deve ser lembrada
antes de comparar recalls de teste entre modelos.

In [18]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

pred = best.predict(X_test)                 # o GridSearchCV já reajustou o vencedor em todo o X_train
proba = best.predict_proba(X_test)[:, 1]
print(f"teste: acurácia {accuracy_score(y_test, pred):.3f} · recall {recall_score(y_test, pred):.3f}"
      f" · ROC AUC {roc_auc_score(y_test, proba):.3f}")
print(confusion_matrix(y_test, pred, labels=[1, 0]))

teste: acurácia 0.982 · recall 0.976 · ROC AUC 0.997
[[41  1]
 [ 1 71]]


**Resposta (depois de rodar):** O teste deu acurácia **0,982**, recall **0,976** e ROC AUC
**0,997** — o recall caiu **dentro** da faixa esperada pela CV (0,959 ± 0,030 cobriria até ~0,989)
e até um pouco acima da média da CV, um lembrete de que o teste é só um sorteio e pode vir "para
cima" tanto quanto "para baixo". A matriz de confusão `[[41, 1], [1, 71]]` mostra que, dos 42
malignos reais no teste, só **1** foi perdido — e 1/42 ≈ 2,4 p.p., exatamente a granularidade
prevista.

**Simulação: quanto o próprio ajuste infla o número?** O `best_score_` de 0,959 é o melhor de 26 configurações, medidas nos mesmos folds que o escolheram. Avalie **a busca da logística** (as 26 configurações) com validação cruzada aninhada — `cross_val_score(search_lr, ...)` num `cv` externo — e compare os três números: busca, CV aninhada e teste.

In [19]:
nested_scores = cross_val_score(search_lr, X_train, y_train, cv=cv, scoring='recall')
print(f"busca (best_score_):   {search_lr.best_score_:.3f}")
print(f"CV aninhada (5 folds):  {nested_scores.mean():.3f} ± {nested_scores.std():.3f}  "
      f"(escores: {np.round(nested_scores, 3)})")
print(f"teste (Passo 12):       0.976")

busca (best_score_):   0.959
CV aninhada (5 folds):  0.947 ± 0.022  (escores: [0.912 0.971 0.941 0.941 0.971])
teste (Passo 12):       0.976


**Resposta:** A busca da logística (26 configurações, medidas nos mesmos folds que a
escolheram) reporta `best_score_` = **0,959**. Avaliando essa mesma busca com CV aninhada — um
`cv` externo em volta do `GridSearchCV` — a estimativa honesta cai para **0,947 ± 0,022**: cerca de
1,2 ponto percentual abaixo do `best_score_`, confirmando que escolher a melhor entre muitas
configurações nos mesmos folds que as mediu é, sim, um pouco otimista (a "maldição do vencedor").
O teste (0,976) ficou **acima** dos dois — um recordatório de que um único teste pode ir para
qualquer lado; o número que deveria orientar a expectativa para dados novos é o da CV aninhada
(0,947), não o `best_score_` nem o acaso favorável do teste.

## Passo 13 — o que o modelo aprendeu?

**Preveja.** Os atributos estão padronizados, então os coeficientes são comparáveis. Você espera que os maiores sejam medidas de tamanho, de textura ou de forma? E eles seriam estáveis se o modelo fosse reajustado em outro fold?

**Resposta (antes de rodar):** Como os 30 atributos foram padronizados, os coeficientes já
são comparáveis em escala, e eu esperaria que medidas de **textura e forma** (não só tamanho)
aparecessem entre as mais importantes, porque o diagnóstico de malignidade depende de irregularidade
de contorno tanto quanto de volume. Não esperaria que fossem estáveis entre folds: com 30 atributos
correlacionados entre si (ex.: `radius`, `area` e `perimeter` medem quase a mesma coisa), a
regressão logística pode distribuir o "crédito" de forma diferente a cada reajuste.

In [20]:
coef = pd.Series(best[-1].coef_[0], index=X.columns)
print(coef.sort_values(key=abs, ascending=False).head(6).round(2))

worst texture          0.96
radius error           0.80
worst symmetry         0.72
worst radius           0.68
mean concave points    0.64
worst area             0.64
dtype: float64


**Resposta (depois de rodar):** Os seis maiores em valor absoluto foram **worst texture**
(0,96), **radius error** (0,80), **worst symmetry** (0,72), **worst radius** (0,68), **mean
concave points** (0,64) e **worst area** (0,64) — uma mistura de textura, forma e tamanho, como
previsto (não só tamanho). Esses números batem exatamente com os da página. Como há colunas
altamente correlacionadas (`radius`, `area`, `perimeter` nas três variantes mean/se/worst), os
coeficientes individuais tendem a ser instáveis — o modelo pode "dividir o crédito" de forma
diferente entre atributos quase redundantes se reajustado em outro fold, mesmo que a *previsão*
final mude pouco.

**Para fechar a aula.** Em uma frase cada: (1) qual dos cinco vazamentos nenhum esquema de CV consegue pegar? (2) Por que comparar fold a fold enxerga diferenças que duas médias ± desvio escondem? (3) Por que o `best_score_` de uma busca em grade é otimista?

**Resposta:**
1. Nenhum esquema de CV pega o vazamento do **Passo 1** (rótulo invertido/trocado) nem, de forma
   geral, qualquer erro de **rotulagem ou de definição do problema** feito antes dos dados
   chegarem à CV — a validação cruzada reparte dados que já estão errados, ela não audita o
   significado do rótulo. O mesmo vale para vazamento por *atributos que conhecem o futuro*: se a
   coluna em si contém informação indevida, repartir os dados de qualquer jeito não resolve.
2. Porque duas médias ± desvio (0,959 ± 0,030 vs. 0,935 ± 0,039) podem parecer "quase iguais" ou
   "claramente diferentes" dependendo de como se olha, mas comparar fold a fold (Passo 9) revelou
   que o sinal da diferença troca (a logística perdeu no fold 2) — informação que as médias
   isoladas escondem porque elas não dizem se os dois modelos erram nos *mesmos* folds difíceis ou
   em folds diferentes.
3. Porque o `best_score_` é o máximo entre várias configurações (26, no caso), medido exatamente
   nos mesmos folds que o escolheram — como visto na simulação do Passo 12, a CV aninhada (0,947)
   ficou abaixo do `best_score_` (0,959): escolher o melhor entre muitos sempre tem uma chance de
   acertar por sorte de amostragem, não só por ser realmente melhor.

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Limiar.** Use `cross_val_predict(best, X_train, y_train, cv=cv, method='predict_proba')[:, 1]` para obter probabilidades fora do fold. Encontre o maior limiar que dá recall ≥ 0,98 e reporte a precisão que você paga por ele. Depois aplique esse limiar, uma vez, ao conjunto de teste.
2. **Penalidade L1.** Troque por `LogisticRegression(l1_ratio=1, solver='liblinear')` (no scikit-learn anterior à 1.8, escreva `penalty='l1'`) e repita o passo 6. Quantos dos 30 coeficientes são exatamente zero no melhor `C`? O recall muda mais de um desvio-padrão?
3. **A escolha inteira, aninhada.** A simulação do passo 12 aninhou só a busca da logística. Escreva o laço externo você mesmo, como na [seção de CV aninhada](https://insper.github.io/ML/pt/2026.2/classes/validation/#a-maldicao-do-vencedor-de-novo-validacao-cruzada-aninhada), para que ele avalie os passos 7 a 9 juntos: as duas buscas *e* a escolha entre elas. O número cai mais?
4. **Mais candidatos.** Acrescente à tabela do passo 8 um candidato SMOTE + k-NN (um pipeline do `imblearn`, como na [aula de ROC e dados desbalanceados](https://insper.github.io/ML/pt/2026.2/classes/roc-imbalanced/)) ajustado no mesmo `cv` e compare-o com o vencedor fold a fold. Depois da [aula de SVM](https://insper.github.io/ML/pt/2026.2/classes/svm/), volte e acrescente também um `SVC`. Algum deles vence a regressão logística ajustada por mais que a dispersão das diferenças pareadas?

### Item 1 — Limiar

In [21]:
from sklearn.metrics import precision_score

proba_oof = cross_val_predict(best, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

best_threshold = 0.0
for t in np.sort(np.unique(proba_oof)):
    pred_t = (proba_oof >= t).astype(int)
    if recall_score(y_train, pred_t) >= 0.98:
        best_threshold = max(best_threshold, t)

pred_oof_best = (proba_oof >= best_threshold).astype(int)
print(f"maior limiar com recall >= 0.98: {best_threshold:.4f}")
print(f"fora do fold: recall {recall_score(y_train, pred_oof_best):.3f}  ·  "
      f"precisão {precision_score(y_train, pred_oof_best):.3f}")

proba_test_thr = best.predict_proba(X_test)[:, 1]
pred_test_thr = (proba_test_thr >= best_threshold).astype(int)
print(f"teste, mesmo limiar: recall {recall_score(y_test, pred_test_thr):.3f}  ·  "
      f"precisão {precision_score(y_test, pred_test_thr):.3f}")

maior limiar com recall >= 0.98: 0.3145
fora do fold: recall 0.982  ·  precisão 0.928
teste, mesmo limiar: recall 0.976  ·  precisão 0.911


**Resposta:** O maior limiar que garante recall ≥ 0,98 fora do fold é **0,3145** (bem abaixo
de 0,5, como esperado — baixar o limiar é o preço de exigir recall mais alto). Nessa escolha, o
recall fora do fold é **0,982** e a precisão cai para **0,928** (de ~0,98 no limiar 0,5 do Passo
12). Aplicando esse mesmo limiar **uma única vez** ao teste: recall **0,976**, precisão **0,911**
— ambos consistentes com a estimativa fora do fold, mostrando que o limiar escolhido com
`cross_val_predict` generaliza bem para dados nunca vistos.

### Item 2 — Penalidade L1

In [22]:
model_l1 = make_pipeline(StandardScaler(),
                          LogisticRegression(l1_ratio=1, solver='liblinear', max_iter=1000))
grid_l1 = GridSearchCV(model_l1, {'logisticregression__C': np.logspace(-3, 3, 13)},
                       cv=cv, scoring='recall', return_train_score=True)
grid_l1.fit(X_train, y_train)

best_C_l1 = grid_l1.best_params_['logisticregression__C']
coefs_l1 = grid_l1.best_estimator_[-1].coef_[0]
n_zero = int((coefs_l1 == 0).sum())
print(f"melhor C (L1): {best_C_l1:g}  ·  recall {grid_l1.best_score_:.3f}")
print(f"coeficientes exatamente zero: {n_zero} de {len(coefs_l1)}")
print(f"recall L2 (Passo 6, C=1): 0.953 ± 0.040  ·  recall L1: {grid_l1.best_score_:.3f} "
      f"± {pd.DataFrame(grid_l1.cv_results_).loc[grid_l1.best_index_, 'std_test_score']:.3f}")

/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


melhor C (L1): 0.01  ·  recall 0.959
coeficientes exatamente zero: 26 de 30
recall L2 (Passo 6, C=1): 0.953 ± 0.040  ·  recall L1: 0.959 ± 0.030


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


**Resposta:** O melhor `C` com penalidade L1 foi **0,01** (bem menor que o `C=1` da L2 —
L1 precisa de menos regularização *em escala de C* para zerar coeficientes), com recall
**0,959 ± 0,030**. Dos 30 coeficientes, **26 ficaram exatamente zero** — a L1 manteve só 4
atributos informativos. O recall (0,959) ficou a 0,006 do valor da L2 no Passo 6 (0,953), bem
dentro de 1 desvio-padrão (0,030-0,040) de qualquer um dos dois: o recall **não muda mais de um
desvio-padrão**, mesmo descartando 26 das 30 variáveis.

### Item 3 — A escolha inteira, aninhada

In [23]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer_scores = []
winners = []
for tr_idx, te_idx in outer_cv.split(X_train, y_train):
    Xtr, Xte = X_train.iloc[tr_idx], X_train.iloc[te_idx]
    ytr, yte = y_train.iloc[tr_idx], y_train.iloc[te_idx]
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    inner_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    inner_knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
    s_lr = GridSearchCV(inner_lr, {'logisticregression__C': np.logspace(-3, 3, 13),
                                   'logisticregression__class_weight': [None, 'balanced']},
                        cv=inner_cv, scoring='recall').fit(Xtr, ytr)
    s_knn = GridSearchCV(inner_knn, {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21, 31],
                                     'kneighborsclassifier__weights': ['uniform', 'distance']},
                         cv=inner_cv, scoring='recall').fit(Xtr, ytr)

    if s_lr.best_score_ >= s_knn.best_score_:
        winner, wname = s_lr.best_estimator_, 'logística'
    else:
        winner, wname = s_knn.best_estimator_, 'k-NN'
    winners.append(wname)
    outer_scores.append(recall_score(yte, winner.predict(Xte)))

outer_scores = np.array(outer_scores)
print("vencedor em cada fold externo:", winners)
print(f"CV aninhada da escolha inteira: {outer_scores.mean():.3f} ± {outer_scores.std():.3f}  "
      f"(escores: {outer_scores.round(3)})")
print("comparar com: busca simples 0.959 · CV aninhada só da logística (sim. do Passo 12) 0.947")

vencedor em cada fold externo: ['logística', 'logística', 'logística', 'logística', 'logística']
CV aninhada da escolha inteira: 0.947 ± 0.039  (escores: [0.941 0.941 0.882 0.971 1.   ])
comparar com: busca simples 0.959 · CV aninhada só da logística (sim. do Passo 12) 0.947


**Resposta:** Aninhando os Passos 7-9 inteiros (as duas buscas *e* a escolha entre elas) em
5 folds externos, a logística venceu a escolha interna em **todos os 5** folds externos. O recall
honesto dessa escolha inteira foi **0,947 ± 0,039** — praticamente igual à CV aninhada que
aninhava só a busca da logística (0,947, na simulação do Passo 12) e ainda abaixo do
`best_score_` ingênuo (0,959). O número **não cai mais** por aninhar a escolha entre famílias
também — nesse caso, porque a logística já vencia de forma consistente, então adicionar a
incerteza "qual família vence" não piora a estimativa.

### Item 4 — Mais candidatos

In [24]:
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

smote_knn = make_imb_pipeline(StandardScaler(), SMOTE(random_state=42), KNeighborsClassifier())
svc = make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))

extra_candidates = {'logistic, tuned (vencedor)': best, 'SMOTE + k-NN': smote_knn, 'SVC': svc}
extra_scores = {name: cross_validate(est, X_train, y_train, cv=cv, scoring=['recall', 'roc_auc'])
                for name, est in extra_candidates.items()}
for name, s in extra_scores.items():
    r = pd.Series(s['test_recall'])
    print(f"{name:28s} recall {r.mean():.3f} ± {r.std():.3f}  ·  "
          f"ROC AUC {pd.Series(s['test_roc_auc']).mean():.3f}")

for name in ['SMOTE + k-NN', 'SVC']:
    d = (pd.Series(extra_scores['logistic, tuned (vencedor)']['test_recall'])
         - pd.Series(extra_scores[name]['test_recall']))
    print(f"vencedor - {name}: diferença média {d.mean():+.3f} ± {d.std():.3f}")

logistic, tuned (vencedor)   recall 0.959 ± 0.034  ·  ROC AUC 0.995
SMOTE + k-NN                 recall 0.965 ± 0.025  ·  ROC AUC 0.988
SVC                          recall 0.947 ± 0.038  ·  ROC AUC 0.995
vencedor - SMOTE + k-NN: diferença média -0.006 ± 0.044
vencedor - SVC: diferença média +0.012 ± 0.034


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/e

**Resposta:** O `SMOTE + k-NN` teve recall **0,965 ± 0,025**, ligeiramente **acima** do
vencedor (logística tunada, 0,959 ± 0,034); o `SVC` teve recall **0,947 ± 0,038**, ligeiramente
abaixo. Mas em ambos os casos a diferença pareada com o vencedor (−0,006 ± 0,044 contra o
SMOTE+k-NN; +0,012 ± 0,034 contra o SVC) fica bem dentro do desvio da diferença — ou seja,
**nenhum** dos dois novos candidatos vence a logística tunada por mais do que a dispersão das
diferenças fold a fold. A regra de "empate, prefira o mais simples" (seção 5 da página) continua
favorecendo a regressão logística tunada.

## Demonstração extra — a taxonomia de vazamento de dados

**Nota sobre esta seção (adicionada na resolução, não está no notebook oficial do professor).**
A página da aula descreve, na seção *"4. Data leakage"*, cinco tipos de vazamento — (1)
pré-processamento antes do split, (2) atributos que conhecem o futuro, (3) linhas duplicadas ou
quase-duplicadas, (4) vazamento por grupo e (5) vazamento temporal — com demonstrações próprias
(incluindo uma medida em "ruído puro" e um experimento completo com mamografias/SMOTE na seção
"6. A leak-free experiment"). O notebook oficial (`_originais/lab-validation.ipynb`) **não**
inclui células de código para essas demonstrações: os 39 passos do lab cobrem só a parte
"Fundamental lab" (Passos 1-6) e "Advanced lab" (Passos 7-13), que tratam de CV e comparação de
modelos, não da taxonomia de vazamento em si.

Como esta aula se chama exatamente *"Cross-Validation and Data Leakage"*, e vazamento de dados é
o próprio assunto da aula, reproduzimos abaixo versões simplificadas, com dados sintéticos
próprios, de três dos cinco tipos (pré-processamento antes do split, duplicatas/vazamento por
grupo, e vazamento temporal). Os números aqui **não são os mesmos da página** (a página usa seus
próprios datasets ilustrativos, que não baixamos) — o objetivo é reproduzir o *fenômeno*, não os
dígitos exatos. Isso também cobre o espírito do item 3 da seção "7. Exercises" da página
(vazamento por seleção de atributos em ruído) e, de forma substituta, do item 2 (vazamento
temporal — a página usa uma série do IBOVESPA que não está disponível neste ambiente; usamos uma
série sintética autocorrelacionada no lugar, documentando a divergência). O tipo (2), "atributos
que conhecem o futuro", é mais conceitual do que demonstrável com um dataset genérico (depende de
qual coluna, em qual problema real, carrega informação do desfecho) — fica só descrito em texto.

### Extra A — pré-processamento antes do split, medido em ruído puro

Geramos 200 linhas de ruído puro (nenhuma relação real com `y`) e 300 colunas também de ruído.
Comparamos selecionar as 20 melhores colunas (`SelectKBest`) **antes** da CV (vendo inclusive os
dados que serão validados) com selecioná-las **dentro** do pipeline (refeito em cada fold).

In [25]:
from sklearn.feature_selection import SelectKBest, f_classif

rng_noise = np.random.RandomState(0)
n_noise, p_noise = 200, 300
X_noise = pd.DataFrame(rng_noise.normal(size=(n_noise, p_noise)),
                       columns=[f'ruido_{i}' for i in range(p_noise)])
y_noise = pd.Series(rng_noise.randint(0, 2, size=n_noise))

cv_noise = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# vazamento: seleciona as colunas usando TODOS os dados (inclusive os que serão validados)
selector_leaky = SelectKBest(f_classif, k=20).fit(X_noise, y_noise)
X_selected = X_noise.iloc[:, selector_leaky.get_support(indices=True)]
leaky_scores = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise,
                               cv=cv_noise, scoring='accuracy')

# honesto: seleção refeita dentro de cada fold
honest_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
honest_scores = cross_val_score(honest_pipe, X_noise, y_noise, cv=cv_noise, scoring='accuracy')

print(f"vazamento (seleção fora do CV):      acurácia {leaky_scores.mean():.3f} ± {leaky_scores.std():.3f}")
print(f"honesto (seleção dentro do pipeline): acurácia {honest_scores.mean():.3f} ± {honest_scores.std():.3f}")
print("acaso puro (50/50, 2 classes): 0.500")

vazamento (seleção fora do CV):      acurácia 0.680 ± 0.102
honesto (seleção dentro do pipeline): acurácia 0.475 ± 0.047
acaso puro (50/50, 2 classes): 0.500


**Resposta:** Selecionar as 20 colunas "mais informativas" usando todos os dados antes da CV
deu acurácia **0,680 ± 0,102** — bem acima do acaso (0,500) **mesmo que os dados sejam ruído
puro**, porque a seleção "espiou" exatamente os dados que depois seriam usados para validar, e
achou por puro acaso 20 colunas entre 300 que pareciam correlacionadas com `y` nessa amostra
específica. Refazendo a seleção dentro do pipeline (uma vez por fold, só com os dados de treino
daquele fold), a acurácia caiu para **0,475 ± 0,047** — dentro do esperado para ruído. A diferença
de ~20 pontos percentuais existe inteiramente por vazamento, sem nenhuma informação real
envolvida — o mesmo mecanismo da seção 4.1 da página ("preprocessing before splitting") e do
item 3 da seção "Exercises".

### Extra B — linhas quase-duplicadas e vazamento por grupo

Um modelo que *memoriza* exemplos (aqui, k-NN com `k=1`) é especialmente vulnerável a
quase-duplicatas: se um "gêmeo" quase idêntico de uma linha de treino cai no fold de validação, o
modelo "reconhece" o gêmeo em vez de generalizar. Comparamos `KFold` comum (que ignora que as
linhas são gêmeas) com `GroupKFold` (que mantém cada linha e seu gêmeo sempre no mesmo fold).

In [26]:
from sklearn.datasets import make_classification
from sklearn.model_selection import GroupKFold

rng_dup = np.random.RandomState(0)
Xd, yd = make_classification(n_samples=300, n_features=20, n_informative=4, n_redundant=0,
                             flip_y=0.35, class_sep=0.6, random_state=0)
Xd = pd.DataFrame(Xd, columns=[f'x{i}' for i in range(20)])
yd = pd.Series(yd)

model_1nn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
cv_clean_dup = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
clean_dup = cross_val_score(model_1nn, Xd, yd, cv=cv_clean_dup, scoring='accuracy')

# duplica TODAS as linhas como "gêmeos quase idênticos" (ruído desprezível)
dup_idx = rng_dup.choice(Xd.index, size=len(Xd), replace=False)
X_twins = Xd.loc[dup_idx] + rng_dup.normal(scale=1e-6, size=(len(dup_idx), Xd.shape[1]))
y_twins = yd.loc[dup_idx]
X_dup = pd.concat([Xd, X_twins], ignore_index=True)
y_dup = pd.concat([yd.reset_index(drop=True), y_twins.reset_index(drop=True)], ignore_index=True)
groups_dup = np.concatenate([Xd.index.values, dup_idx])

leaky_dup = cross_val_score(model_1nn, X_dup, y_dup,
                            cv=KFold(n_splits=5, shuffle=True, random_state=0), scoring='accuracy')
honest_dup = cross_val_score(model_1nn, X_dup, y_dup,
                             cv=GroupKFold(n_splits=5), groups=groups_dup, scoring='accuracy')

print(f"sem duplicatas (1-NN):                      acurácia {clean_dup.mean():.3f} ± {clean_dup.std():.3f}")
print(f"KFold ignorando os gêmeos (100% duplicado):  acurácia {leaky_dup.mean():.3f} ± {leaky_dup.std():.3f}")
print(f"GroupKFold (gêmeos sempre no mesmo fold):     acurácia {honest_dup.mean():.3f} ± {honest_dup.std():.3f}")

sem duplicatas (1-NN):                      acurácia 0.617 ± 0.021
KFold ignorando os gêmeos (100% duplicado):  acurácia 0.927 ± 0.023
GroupKFold (gêmeos sempre no mesmo fold):     acurácia 0.600 ± 0.055


**Resposta:** Sem duplicatas, o 1-NN tem acurácia **0,617 ± 0,021** (um 1-NN não é um
classificador forte neste problema ruidoso). Duplicando todas as linhas como gêmeos quase
idênticos e usando `KFold` comum, a acurácia **sobe para 0,927 ± 0,023** — o modelo está, na
prática, "copiando a resposta" do seu próprio gêmeo quando ele cai no fold de treino e o original
cai no de validação (ou vice-versa). Usando `GroupKFold` para manter cada par sempre no mesmo
fold, a acurácia volta a **0,600 ± 0,055**, igual à linha de base sem duplicatas — confirmando que
o salto de 30 pontos percentuais era vazamento puro, não melhora real. Esse é o mesmo mecanismo
das seções 4.3 ("Duplicate or near-duplicate rows") e 4.4 ("Group leakage") da página — aqui
combinadas no mesmo experimento, já que a solução (agrupar os duplicados) é a mesma.

### Extra C — vazamento temporal

A página usa uma série de preços do IBOVESPA para mostrar que embaralhar uma série temporal antes
da CV é desonesto — essa série não está disponível neste ambiente, então substituímos por uma
série sintética autocorrelacionada (passeio aleatório com tendência), documentando a divergência.
Comparamos `KFold` embaralhado com `TimeSeriesSplit` (que só valida em dados *depois* do treino).

In [27]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression

rng_ts = np.random.RandomState(0)
n_ts = 300
trend = 0.05 * np.arange(n_ts)
passeio = np.cumsum(rng_ts.normal(scale=1.0, size=n_ts))
serie = trend + passeio

df_ts = pd.DataFrame({'y': serie})
for lag in [1, 2, 3]:
    df_ts[f'lag{lag}'] = df_ts['y'].shift(lag)
df_ts = df_ts.dropna().reset_index(drop=True)
X_ts, y_ts = df_ts[['lag1', 'lag2', 'lag3']], df_ts['y']

model_ts = make_pipeline(StandardScaler(), LinearRegression())

r2_shuffled = cross_val_score(model_ts, X_ts, y_ts,
                              cv=KFold(n_splits=5, shuffle=True, random_state=0), scoring='r2')
r2_honest = cross_val_score(model_ts, X_ts, y_ts, cv=TimeSeriesSplit(n_splits=5), scoring='r2')

print(f"KFold embaralhado (vê o 'futuro' via vizinhos):  R² {r2_shuffled.mean():.3f} ± {r2_shuffled.std():.3f}")
print(f"TimeSeriesSplit (só valida depois do treino):     R² {r2_honest.mean():.3f} ± {r2_honest.std():.3f}")

KFold embaralhado (vê o 'futuro' via vizinhos):  R² 0.982 ± 0.005
TimeSeriesSplit (só valida depois do treino):     R² 0.853 ± 0.061


**Resposta:** O `KFold` embaralhado deu R² **0,982 ± 0,005** — muito alto, porque ao
embaralhar a série, cada ponto de validação quase sempre tem um vizinho temporal (defasado só por
1-3 passos) no treino, então o modelo "cola" nos valores vizinhos. O `TimeSeriesSplit` honesto, que
só usa o passado para prever o futuro, deu R² **0,853 ± 0,061** — bem mais baixo, porque numa série
em passeio aleatório o futuro genuinamente diverge do padrão visto até agora. A direção do
resultado é a mesma da página (CV embaralhada otimista, CV temporal honesta mais baixa); a página
chega a um R² negativo na versão honesta, o que indica uma série com estrutura diferente da nossa
(provavelmente com uma quebra de regime que o passeio aleatório sintético não reproduz) — a
magnitude exata diverge porque usamos séries diferentes, mas a lição (nunca embaralhe uma série
temporal) se confirma igual.

## Conclusões

- Um único hold-out 80/20 variou de **0,934 a 1,000** de acurácia entre 20 sementes (amplitude de
  6,6 p.p.), contra só **0,969 a 0,980** (1,1 p.p.) para a média de uma CV 5-fold — a validação
  cruzada troca a loteria de um único split por uma média mais estável.
- A regressão logística tunada (recall **0,959 ± 0,030**) e o k-NN tunado (**0,935 ± 0,039**)
  ficaram em empate técnico: a diferença pareada fold a fold foi **+0,024 ± 0,047** (Passo 9) e
  **+0,029 ± 0,041** em 50 pares repetidos — em ambos os casos o desvio supera a média da
  diferença.
- A busca em grade reportou `best_score_` = **0,959**, mas a CV aninhada (que trata a própria
  busca como parte do modelo a validar) deu uma estimativa mais honesta de **0,947 ± 0,022** —
  cerca de 1,2 p.p. mais baixa, a "maldição do vencedor" medida em números.
- O teste único (Passo 12) deu recall **0,976**, dentro da faixa esperada pela CV e até um pouco
  acima dela — com só 42 malignos no teste, cada caso vale **2,4 pontos percentuais** de recall,
  então comparações finas entre modelos no teste têm pouca resolução.
- Vazamento por seleção de atributos em ruído puro inflou a acurácia de 0,475 (honesto) para
  0,680 (vazado); vazamento por duplicatas/grupo inflou de 0,600 para 0,927 num 1-NN; vazamento
  temporal (CV embaralhada) deu R² 0,982 contra 0,853 da versão honesta — em todos os casos a
  célula de código que decide "o que o modelo nunca viu" é o que separa um número real de um
  número inventado pelo próprio esquema de validação.
- Regra prática que atravessa o lab inteiro: o `StandardScaler`, a seleção de atributos e a escolha
  de hiperparâmetros têm que **recomeçar em cada fold** — a única vez em que isso não se aplica é
  ao `X_test`, trancado do Passo 2 até o Passo 12 e usado exatamente uma vez.